<a href="https://colab.research.google.com/github/PrabhuRajendhran/100-days-of-inference/blob/main/vLLM_setup_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Kill any hidden background processes trying to run the old setup
!pkill -f vllm

# 2. Forcefully remove the incompatible build and clean the pip cache
!pip uninstall -y vllm vllm-flash-attn
!pip cache purge

# 3. Install a stable vLLM version compiled for CUDA 12
!pip install vllm==0.19.1 --extra-index-url https://pytorch.org

Found existing installation: vllm 0.19.1
Uninstalling vllm-0.19.1:
  Successfully uninstalled vllm-0.19.1
Files removed: 118
Looking in indexes: https://pypi.org/simple, https://pytorch.org
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.1/433.1 MB 1.9 MB/s eta 0:00:00


In [2]:
!nohup python3 -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen2.5-0.5B-Instruct \
    --port 8000 \
    --dtype half \
    --gpu-memory-utilization 0.8 \
    > vllm.log 2>&1 &

In [8]:
!tail -n 20 vllm.log

(APIServer pid=9172) INFO 06-23 03:29:39 [utils.py:299]        █     █     █▄   ▄█
(APIServer pid=9172) INFO 06-23 03:29:39 [utils.py:299]  ▄▄ ▄█ █     █     █ ▀▄▀ █  version 0.19.1
(APIServer pid=9172) INFO 06-23 03:29:39 [utils.py:299]   █▄█▀ █     █     █     █  model   Qwen/Qwen2.5-0.5B-Instruct
(APIServer pid=9172) INFO 06-23 03:29:39 [utils.py:299]    ▀▀  ▀▀▀▀▀ ▀▀▀▀▀ ▀     ▀
(APIServer pid=9172) INFO 06-23 03:29:39 [utils.py:299] 
(APIServer pid=9172) INFO 06-23 03:29:39 [utils.py:233] non-default args: {'model': 'Qwen/Qwen2.5-0.5B-Instruct', 'dtype': 'half', 'gpu_memory_utilization': 0.8}
(APIServer pid=9172) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
(APIServer pid=9172) INFO 06-23 03:29:57 [model.py:549] Resolved architecture: Qwen2ForCausalLM
(APIServer pid=9172) WARNING 06-23 03:29:57 [model.py:2016] Casting torch.bfloat16 to torch.float16.
(APIServer pid=9172) INFO 06-23 03:29:57 

In [4]:
!ps aux | grep vllm

root        9172  8.0  0.2 852524 33268 ?        D    03:29   0:00 python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-0.5B-Instruct --port 8000 --dtype half --gpu-memory-utilization 0.8
root        9177  0.0  0.0   7376  3480 ?        S    03:29   0:00 /bin/bash -c ps aux | grep vllm
root        9179  0.0  0.0   1184   348 ?        D    03:29   0:00 grep vllm


In [10]:
!curl http://localhost:8000/v1/models

curl: (7) Failed to connect to localhost port 8000 after 0 ms: Connection refused


In [ ]:
import json
import urllib.request

# Define your prompt and structural payload
url = "http://localhost:8000/v1/chat/completions"
data = {
    "model": "Qwen/Qwen2.5-0.5B-Instruct",
    "messages": [
        {"role": "system", "content": "You are a helpful, concise AI assistant."},
        {"role": "user", "content": "Explain what a GPU is in exactly one short sentence."}
    ],
    "temperature": 0.7,
    "max_tokens": 50
}

# Process the direct API network request
req = urllib.request.Request(
    url,
    data=json.dumps(data).encode("utf-8"),
    headers={"Content-Type": "application/json"}
)

try:
    with urllib.request.urlopen(req) as response:
        result = json.loads(response.read().decode("utf-8"))
        print("Model Response:\n", result["choices"][0]["message"]["content"])
except Exception as e:
    print(f"Error connecting to server: {e}. Please wait a few seconds and try again.")
